# OC14 — Quick eval of the SFT (Base) model

Loads the LoRA adapter from the SFT kernel's output and generates with the **correct**
inference config: stop on `<|im_end|>` (the small model otherwise runs past the answer into
repetition), and the **exact trained system prompt** (read back from the dataset). Scores the
held-out hand-labelled triage vignettes. Small set (6) — a sanity check, not the final eval.

In [ ]:
# Official Unsloth Kaggle install (verbatim, June 2026): install torch from the cu128
# index FIRST so its wheels carry the T4/sm_75 CUDA kernels. Plain `pip install unsloth`
# let pip resolve a torch build lacking them -> 'CUDA: no kernel image' at model load.
!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
import subprocess, sys
with open('/kaggle/working/requirements-train.lock.txt', 'w') as fh:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=fh)
print('lockfile written ->', '/kaggle/working/requirements-train.lock.txt')

In [ ]:
import glob, os, json
# Prefer the SFT+DPO adapter if attached; else the SFT adapter. (Compare both runs.)
ad = (glob.glob('/kaggle/input/**/dpo_adapter/adapter_config.json', recursive=True)
      or glob.glob('/kaggle/input/**/sft_adapter/adapter_config.json', recursive=True))
assert ad, 'adapter not found — attach the SFT and/or DPO kernel as a kernel source'
ADAPTER_DIR = os.path.dirname(ad[0]); print('ADAPTER_DIR =', ADAPTER_DIR)
dd = glob.glob('/kaggle/input/**/sft_train.jsonl', recursive=True)
DATA_DIR = os.path.dirname(dd[0]) if dd else None
# Read the EXACT trained system prompts back from the data (guarantees eval matches training).
SYS = {}
if DATA_DIR:
    for ln in open(f'{DATA_DIR}/sft_train.jsonl', encoding='utf-8').read().split('\n'):
        if not ln.strip():
            continue
        r = json.loads(ln)
        SYS.setdefault(r['lang'], r['messages'][0]['content'])
        if len(SYS) == 2:
            break
print('system prompts loaded for langs:', list(SYS))

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_DIR, max_seq_length=2048, load_in_4bit=True)
FastLanguageModel.for_inference(model)
IM_END = tokenizer.convert_tokens_to_ids('<|im_end|>')
STOP_IDS = [IM_END, tokenizer.eos_token_id]
print('eos:', tokenizer.eos_token, '| im_end id:', IM_END)
def gen(system, user):
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': user}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt'
                                        ).to(model.device)
    out = model.generate(input_ids=ids, max_new_tokens=256, do_sample=True,
                         temperature=0.3, top_p=0.9, repetition_penalty=1.1,
                         eos_token_id=STOP_IDS)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

In [ ]:
LEVELS = ('urgence maximale', 'urgence modérée', 'urgence différée')
def extract_urgency(t):
    low = t.lower(); hits = [(low.index(l), l) for l in LEVELS if l in low]
    return min(hits)[1] if hits else None
def has_disclaimer(t):
    low = t.lower(); return ('ne remplace pas' in low) or ('does not replace' in low)
# Held-out hand-labelled vignettes (from vignettes.py, split='eval').
VIGN = [
  ('fr', 'Homme de 60 ans, malaise avec perte de connaissance brève et pâleur intense.', 'urgence maximale'),
  ('fr', 'Adolescent, mal de gorge et fièvre légère depuis hier, sans difficulté à avaler.', 'urgence modérée'),
  ('fr', 'Femme de 35 ans, demande de certificat médical pour le sport.', 'urgence différée'),
  ('en', '50-year-old, sudden worst-ever headache with vomiting and neck stiffness.', 'urgence maximale'),
  ('en', '27-year-old with mild ankle pain after jogging, able to walk.', 'urgence modérée'),
  ('en', '40-year-old asking how to renew a stable long-term prescription.', 'urgence différée'),
]
res = []
for lang, user, gold in VIGN:
    txt = gen(SYS.get(lang, 'You are a medical triage assistant.'), user)
    pred = extract_urgency(txt)
    res.append((lang, gold, pred, has_disclaimer(txt), txt))
    print(f'[{lang}] gold={gold} | pred={pred}')
    print('  ' + txt.replace(chr(10), ' ')[:300]); print('-' * 70)
n = len(res)
acc = sum(g == p for _, g, p, _, _ in res) / n
fmt = sum(p is not None for _, _, p, _, _ in res) / n
disc = sum(d for *_, d, _ in res) / n
print(f'\n=== n={n} | urgency_accuracy={acc:.2f} | format_rate={fmt:.2f} | disclaimer_rate={disc:.2f}')